In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

def genera_dashboard_interattiva(file_csv="report_candidature_definitivo.csv"):
    if not os.path.exists(file_csv):
        print(f"Errore: Il file {file_csv} non è stato trovato.")
        return

    # 1. Caricamento e Pulizia (Senza Warning Pandas)
    df = pd.read_csv(file_csv)
    df_analisi = df[~df['Stato Classificato'].str.contains('Ignorato', na=False)].copy()
    
    df_analisi['Data'] = pd.to_datetime(df_analisi['Data'], errors='coerce', utc=True, format='mixed')
    df_analisi['Mese'] = df_analisi['Data'].dt.tz_localize(None).dt.to_period('M').dt.to_timestamp()
    
    # Calcolo KPI
    tot_candidature = len(df_analisi[df_analisi['Stato Classificato'] == 'Candidatura Ricevuta'])
    tot_colloqui = len(df_analisi[df_analisi['Stato Classificato'].str.contains('Colloquio', na=False)])
    conversion_rate = (tot_colloqui / tot_candidature * 100) if tot_candidature > 0 else 0

    # 2. Creazione della Struttura Interattiva (Griglia 2x2)
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Esito Generale delle Candidature",
            "Volume Comunicazioni nel Tempo",
            "Top 10 Aziende per Interazioni",
            "Aziende con Colloqui Fissati"
        ),
        horizontal_spacing=0.15,
        vertical_spacing=0.15
    )

    # -- Grafico 1: Esito Generale --
    # In Plotly, per avere il valore più alto in cima su un bar chart orizzontale, ordiniamo in modo crescente
    stati_counts = df_analisi['Stato Classificato'].value_counts().sort_values(ascending=True)
    fig.add_trace(go.Bar(
        x=stati_counts.values,
        y=stati_counts.index,
        orientation='h',
        marker=dict(color='#3498db'), # Blu moderno
        text=stati_counts.values,
        textposition='auto',
        hoverinfo='y+x'
    ), row=1, col=1)

    # -- Grafico 2: Andamento nel Tempo --
    andamento = df_analisi.groupby('Mese').size().reset_index(name='Conteggio')
    fig.add_trace(go.Scatter(
        x=andamento['Mese'],
        y=andamento['Conteggio'],
        mode='lines+markers',
        line=dict(color='#2ecc71', width=3), # Verde acceso
        marker=dict(size=10, color='#27ae60'),
        hovertemplate='%{x|%b %Y}<br>Email: %{y}<extra></extra>'
    ), row=1, col=2)

    # -- Grafico 3: Top 10 Aziende --
    top_aziende = df_analisi['Azienda'].value_counts().head(10).sort_values(ascending=True)
    fig.add_trace(go.Bar(
        x=top_aziende.values,
        y=top_aziende.index,
        orientation='h',
        marker=dict(color='#9b59b6'), # Viola
        text=top_aziende.values,
        textposition='auto'
    ), row=2, col=1)

    # -- Grafico 4: Focus Colloqui --
    df_colloqui = df_analisi[df_analisi['Stato Classificato'].str.contains("Colloquio", na=False)]
    if not df_colloqui.empty:
        colloqui_aziende = df_colloqui['Azienda'].value_counts().head(8).sort_values(ascending=True)
        fig.add_trace(go.Bar(
            x=colloqui_aziende.values,
            y=colloqui_aziende.index,
            orientation='h',
            marker=dict(color='#e67e22'), # Arancione
            text=colloqui_aziende.values,
            textposition='auto'
        ), row=2, col=2)
        # Forza l'asse X a mostrare solo numeri interi per i colloqui
        fig.update_xaxes(dtick=1, row=2, col=2) 
    else:
        # Se non ci sono colloqui, mostriamo un grafico vuoto con un messaggio
        fig.add_annotation(
            text="Nessun colloquio registrato",
            xref="x4", yref="y4",
            showarrow=False, font=dict(size=16, color="gray")
        )

    # 3. Ottimizzazione Layout Globale
    fig.update_layout(
        title_text=f"<b>Dashboard Job Tracker Analytics</b><br><sup>Tasso di conversione a colloquio: <b>{conversion_rate:.1f}%</b></sup>",
        title_font_size=24,
        title_x=0.5, # Centra il titolo principale
        height=900,  # Altezza generosa per respirare
        showlegend=False,
        template="plotly_white", # Sfondo bianco pulito
        font=dict(family="Arial, sans-serif", size=12),
        margin=dict(t=120, b=50, l=50, r=50) # Margini regolati per evitare sovrapposizioni
    )

    # Rimuovi il titolo dell'asse Y per pulizia (i nomi sono già autoesplicativi)
    fig.update_yaxes(title_text="", showline=True, linewidth=1, linecolor='lightgray')
    fig.update_xaxes(showline=True, linewidth=1, linecolor='lightgray', gridcolor='whitesmoke')

    # 4. Esportazione e Apertura Automatica
    output_file = "dashboard_interattiva.html"
    fig.write_html(output_file, auto_open=True)
    print(f"\n✅ Dashboard interattiva generata con successo!")
    print(f"File salvato in: {os.path.abspath(output_file)}")

if __name__ == "__main__":
    genera_dashboard_interattiva()